In [1]:
!pip install langchain-experimental langchain-openai pandas requests==2.32.4
!pip install gdown

In [9]:
import pandas as pd
import os
import sys
import gdown
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# ==========================================
#  PART 1: AUTOMATIC FILE DOWNLOADER
# ==========================================

files_to_download = {
    "saas_docs.csv":         "https://drive.google.com/file/d/1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq/view?usp=sharing",
    "credit_card_terms.csv": "https://drive.google.com/file/d/1_giivc_B0urOKpct0XY2yVZuxW3Eenuf/view?usp=sharing",
    "hospital_policy.csv":   "https://drive.google.com/file/d/1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm/view?usp=sharing",
    "ecommerce_faqs.csv":    "https://drive.google.com/file/d/1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p/view?usp=sharing"
}

print("--- Downloading Files from Google Drive ---")
for filename, url in files_to_download.items():
    if not os.path.exists(filename):
        gdown.download(url, filename, quiet=False, fuzzy=True)
        print(f"Downloaded: {filename}")
    else:
        print(f"Skipped: {filename} (Already exists)")
print("--- Download Complete ---")


# ==========================================
#  PART 2: AI AGENT SETUP (MULTI-FILE)
# ==========================================

# 1. SETUP: Get API Key Securely
print("ENTER YOUR OPENAI API KEY BELOW:")
api_key = getpass()

# 2. LOAD ALL CSVs INTO A LIST
dataframes = [] # We will store all the loaded tables here
loaded_names = []

try:
    for filename in files_to_download.keys():
        df = pd.read_csv(filename)
        dataframes.append(df)
        loaded_names.append(filename)
        print(f"SUCCESS: Loaded '{filename}' ({len(df)} rows)")

except Exception as e:
    print(f"\nERROR loading files: {e}")
    sys.exit()

# 3. DEFINE THE RULES
system_prompt = """
You are a smart data assistant capable of reading multiple CSV files.
- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.
- When asked a question, determine which DataFrame is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.
"""

try:
    # ---------------------------------------------------------
    # TODO 1: Initialize the LLM
    # Hint: Use ChatOpenAI with model="gpt-4o-mini" and temperature=0.0
    # ---------------------------------------------------------
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0, openai_api_key=api_key)

    # ---------------------------------------------------------
    # TODO 2: Create the Pandas Agent
    # Hint: Pass the 'llm' and the list 'dataframes' as arguments.
    # Set verbose=True and allow_dangerous_code=True
    # ---------------------------------------------------------
    agent = create_pandas_dataframe_agent(
        llm,
        dataframes, # <--- PASS THE DATAFRAMES LIST HERE
        verbose=True,
        agent_type="openai-functions",
        allow_dangerous_code=True
    )

    print("\nAI Agent is ready! You can ask questions across ALL files.")
    print("Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'")

except Exception as e:
    print(f"Error initializing agent: {e}")
    sys.exit()

--- Downloading Files from Google Drive ---
Skipped: saas_docs.csv (Already exists)
Skipped: credit_card_terms.csv (Already exists)
Skipped: hospital_policy.csv (Already exists)
Skipped: ecommerce_faqs.csv (Already exists)
--- Download Complete ---
ENTER YOUR OPENAI API KEY BELOW:
··········
SUCCESS: Loaded 'saas_docs.csv' (15 rows)
SUCCESS: Loaded 'credit_card_terms.csv' (15 rows)
SUCCESS: Loaded 'hospital_policy.csv' (15 rows)
SUCCESS: Loaded 'ecommerce_faqs.csv' (15 rows)

AI Agent is ready! You can ask questions across ALL files.
Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'


In [10]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

In [12]:
import os
import openai
from openai import OpenAI
from google.colab import userdata

# Initialize the OpenAI client
# It will automatically look for OPENAI_API_KEY in your environment variables.
try:
    client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
except Exception as e:
    print(f"no An unexpected error occurred during client initialization: {e}")
    exit()

# --- 2. Define your request parameters ---
model_name = "gpt-4o"  # Or "gpt-4", "gpt-4o", etc.
user_message = "What is the capital of France?"

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": user_message}
]

# Optional: Add more parameters for fine-tuning the response
temperature = 0.7  # Controls randomness. Lower for more deterministic, higher for more creative.
max_tokens = 150   # Maximum number of tokens (words/sub-words) in the response.

print(f"Sending request to model: {model_name}")
print(f"User message: {user_message}\n")

# --- 3. Make the API call ---
try:
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )

    # --- 4. Process the response ---
    if response.choices:
        assistant_reply = response.choices[0].message.content
        print("Assistant's Reply:")
        print(assistant_reply)
    else:
        print("No response choices found.")

except openai.APIConnectionError as e:
    print(f"Could not connect to OpenAI API: {e}")
except openai.RateLimitError as e:
    print(f"OpenAI API rate limit exceeded: {e}")
except openai.APIStatusError as e:
    print(f"OpenAI API returned an error status {e.status_code}: {e.response}")
except openai.AuthenticationError as e:
    print(f"OpenAI API authentication failed: {e}")
    print("Please check your API key.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Sending request to model: gpt-4o
User message: What is the capital of France?

Assistant's Reply:
The capital of France is Paris.


In [14]:
import os
import openai
from openai import OpenAI
from google.colab import userdata


# --- 1. Set your API key ---
# It's highly recommended to set your API key as an environment variable
# named OPENAI_API_KEY. The `openai` library automatically picks it up.
# If you must set it directly (e.g., for quick testing), uncomment the line below,
# but remember this is NOT recommended for production or shared code.
#openai.api_key = getkey

# Initialize the OpenAI client
# It will automatically look for OPENAI_API_KEY in your environment variables.
try:
    client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
except Exception as e:
    print(f"An unexpected error occurred during client initialization: {e}")
    exit()

# --- 2. Define your request parameters ---
model_name = "gpt-4o" #"gpt-3.5-turbo"  # Or "gpt-4", "gpt-4o", etc.
user_message = "What is the capital of France?"

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": user_message}
]

# Optional: Add more parameters for fine-tuning the response
temperature = 0.7  # Controls randomness. Lower for more deterministic, higher for more creative.
max_tokens = 150   # Maximum number of tokens (words/sub-words) in the response.

print(f"Sending request to model: {model_name}")
print(f"User message: {user_message}\n")

# --- 3. Make the API call ---
try:
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )

    # --- 4. Process the response ---
    if response.choices:
        assistant_reply = response.choices[0].message.content
        print("Assistant's Reply:")
        print(assistant_reply)
    else:
        print("No response choices found.")

except openai.APIConnectionError as e:
    print(f"Could not connect to OpenAI API: {e}")
except openai.RateLimitError as e:
    print(f"OpenAI API rate limit exceeded: {e}")
except openai.APIStatusError as e:
    print(f"OpenAI API returned an error status {e.status_code}: {e.response}")
except openai.AuthenticationError as e:
    print(f"OpenAI API authentication failed: {e}")
    print("Please check your API key.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Sending request to model: gpt-4o
User message: What is the capital of France?

Assistant's Reply:
The capital of France is Paris.


In [15]:
# ==========================================
#  PART 3: CHAT LOOP
# ==========================================
print("\nType 'exit' or 'quit' to stop conversation.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input:
        continue

    final_query = system_prompt + "\n\nQuestion: " + user_input
    print("AI is thinking...")

    try:
        # ---------------------------------------------------------
        # TODO 4: Invoke the Agent
        # Hint: Use agent.invoke() and pass the final_query
        # The result will be a dictionary, access ['output']
        # ---------------------------------------------------------
        response = "hi hello yes..." # <--- REPLACE THIS WITH YOUR CODE

        print(f"AI: {response}\n" + "-"*30)
    except Exception as e:
        print(f"An error occurred: {e}")


Type 'exit' or 'quit' to stop conversation.

You: exit
Goodbye!
